# Módulo 49 · DL V: PyTorch y el bucle de entrenamiento

Código del módulo para ejecutar en **Google Colab**, donde TensorFlow y PyTorch ya vienen instalados. Para entrenar más rápido activa la GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. Las explicaciones, los ejercicios y las partes que funcionan en el navegador están en el módulo 49 de la web.

In [ ]:
# Solo si lo ejecutas en tu ordenador (en Colab ya está todo instalado):
# !pip install torch torchvision matplotlib

### PyTorch: red densa con nn.Sequential (Colab)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

model = nn.Sequential(
    nn.Flatten(),                  # 28x28 -> 784
    nn.Linear(784, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 10),
    nn.LogSoftmax(dim=1),          # log-probabilidades...
)
criterion = nn.NLLLoss()           # ...con NLLLoss equivale a nn.CrossEntropyLoss() sin LogSoftmax
optimizer = optim.SGD(model.parameters(), lr=0.01)

for epoch in range(10):
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    print(f"Epoch {epoch + 1}, Loss: {running_loss / len(train_loader):.4f}, Accuracy: {100 * correct / total:.2f}%")

model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        correct += (model(images).argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
print(f"Final Test Accuracy: {100 * correct / total:.2f}%")

### PyTorch: red convolucional con nn.Module, validación y test (Colab)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

device = "cuda" if torch.cuda.is_available() else "cpu"        # en Colab: Entorno de ejecución > GPU
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
trainset = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
testset = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)
n_val = int(0.15 * len(trainset))
train_data, val_data = random_split(trainset, [len(trainset) - n_val, n_val], generator=torch.Generator().manual_seed(42))
trainloader = DataLoader(train_data, batch_size=64, shuffle=True)
valloader = DataLoader(val_data, batch_size=64)
testloader = DataLoader(testset, batch_size=64)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 3 * 3, 128)       # 28 -> 14 -> 7 -> 3 tras tres max-pooling
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)                         # logits: CrossEntropyLoss aplica el softmax

model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def evaluar(loader):
    model.eval()
    perdida, correctos, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            perdida += criterion(outputs, labels).item()
            correctos += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return perdida / len(loader), 100 * correctos / total

for epoch in range(5):
    model.train()
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch + 1}: validación (loss, acc) = {evaluar(valloader)}")
print("Test (loss, acc):", evaluar(testloader))